## Welcome to the Second Lab - Week 1, Day 3

Today we will work with lots of models! This is a way to get comfortable with APIs.

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/stop.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#ff7800;">Important point - please read</h2>
            <span style="color:#ff7800;">The way I collaborate with you may be different to other courses you've taken. I prefer not to type code while you watch. Rather, I execute Jupyter Labs, like this, and give you an intuition for what's going on. My suggestion is that you carefully execute this yourself, <b>after</b> watching the lecture. Add print statements to understand what's going on, and then come up with your own variations.<br/><br/>If you have time, I'd love it if you submit a PR for changes in the community_contributions folder - instructions in the resources. Also, if you have a Github account, use this to showcase your variations. Not only is this essential practice, but it demonstrates your skills to others, including perhaps future clients or employers...
            </span>
        </td>
    </tr>
</table>

In [1]:
# Start with imports - ask ChatGPT to explain any package that you don't know

import os
import json
from dotenv import load_dotenv
from openai import OpenAI
from anthropic import Anthropic
from IPython.display import Markdown, display

In [2]:
# Always remember to do this!
load_dotenv(override=True)

True

In [3]:
# Print the key prefixes to help with any debugging

openai_api_key = os.getenv('OPENAI_API_KEY')
anthropic_api_key = os.getenv('ANTHROPIC_API_KEY')
google_api_key = os.getenv('GOOGLE_API_KEY')
deepseek_api_key = os.getenv('DEEPSEEK_API_KEY')
groq_api_key = os.getenv('GROQ_API_KEY')

if openai_api_key:
    print(f"OpenAI API Key exists and begins {openai_api_key[:8]}")
else:
    print("OpenAI API Key not set")
    
if anthropic_api_key:
    print(f"Anthropic API Key exists and begins {anthropic_api_key[:7]}")
else:
    print("Anthropic API Key not set (and this is optional)")

if google_api_key:
    print(f"Google API Key exists and begins {google_api_key[:2]}")
else:
    print("Google API Key not set (and this is optional)")

if deepseek_api_key:
    print(f"DeepSeek API Key exists and begins {deepseek_api_key[:3]}")
else:
    print("DeepSeek API Key not set (and this is optional)")

if groq_api_key:
    print(f"Groq API Key exists and begins {groq_api_key[:4]}")
else:
    print("Groq API Key not set (and this is optional)")

OpenAI API Key exists and begins sk-proj-
Anthropic API Key not set (and this is optional)
Google API Key not set (and this is optional)
DeepSeek API Key not set (and this is optional)
Groq API Key not set (and this is optional)


In [4]:
request = "Please come up with a challenging, nuanced question that I can ask a number of LLMs to evaluate their intelligence. "
request += "Answer only with the question, no explanation."
messages = [{"role": "user", "content": request}]

In [5]:
messages

[{'role': 'user',
  'content': 'Please come up with a challenging, nuanced question that I can ask a number of LLMs to evaluate their intelligence. Answer only with the question, no explanation.'}]

In [6]:
openai = OpenAI()
response = openai.chat.completions.create(
    model="gpt-5-mini",
    messages=messages,
)
question = response.choices[0].message.content
print(question)


Design a rigorous, practical evaluation (including at least eight concrete test items with example correct outputs) that reliably distinguishes language models that primarily rely on memorized training data from those that demonstrate genuine compositional, generalizable reasoning; your answer should: (1) state the principles the evaluation uses to separate memorization from reasoning and explain why those principles reduce false positives and false negatives, (2) present scoring metrics, decision thresholds, and an overall pass/fail rubric, (3) provide at least eight specific test prompts spanning domains such as multi-step symbolic reasoning, novel analogies, counterfactual and hypothetical reasoning, commonsense physics, planning under constraints, program synthesis using an invented API, and adversarial distractors—include for each prompt the expected correct behavior/output and brief rationale for why that prompt is diagnostic, (4) explain likely failure modes where a memorizing m

In [8]:
competitors = []
answers = []
messages = [{"role": "user", "content": question}]

## Note - update since the videos

I've updated the model names to use the latest models below, like GPT 5 and Claude Sonnet 4.5. It's worth noting that these models can be quite slow - like 1-2 minutes - but they do a great job! Feel free to switch them for faster models if you'd prefer, like the ones I use in the video.

In [9]:
# The API we know well
# I've updated this with the latest model, but it can take some time because it likes to think!
# Replace the model with gpt-4.1-mini if you'd prefer not to wait 1-2 mins

model_name = "gpt-5-nano"

response = openai.chat.completions.create(model=model_name, messages=messages)
answer = response.choices[0].message.content

display(Markdown(answer))
competitors.append(model_name)
answers.append(answer)

Below is a self-contained, rigorous evaluation design intended to distinguish language models that mainly memorize training data from those that exhibit genuine compositional, generalizable reasoning. It covers principles, scoring, eight concrete test items with example outputs, failure modes with countermeasures, a reproducible evaluation protocol, and item-by-item observable differences to aid external researchers.

1) Principles separating memorization from reasoning (and rationale for reducing false positives/negatives)

- Core principle 1: Novelty and combinatoriality
  - Memorization tends to falter on inputs that (a) are not exact replicas of training data, or (b) require novel combinations of known concepts. Genuine reasoning should generalize to new, unseen mixtures of concepts.
  - Why this reduces false positives: If a model only regurgitates memorized strings, it will struggle on prompts that mix previously seen components in a novel way. A reasoning model should produce correct outputs for these novel combinations.
  - Why this reduces false negatives: Reasoning models can adapt to new problem structures even when they have seen similar building blocks before; memorization-based models typically fail when the exact structure or pairing hasn’t appeared in training.

- Core principle 2: Constraint-driven and multi-step processing rather than surface pattern matching
  - Reasoning requires internal manipulation of symbols, constraints, or steps (even if not revealed as chain-of-thought), not just plausible-sounding text templates.
  - Why this reduces false positives: Surface-level templates often produce plausible outputs for many prompts; genuine reasoning requires consistent, rule-based processing (e.g., math, formal grammars, multi-step planning).

- Core principle 3: Cross-domain transfer
  - A genuine generalizing model should transfer reasoning skills across domains (math, physics, planning, programming) without needing to memorize domain-specific templates.
  - Why this reduces false positives: If a model can apply similar reasoning patterns across domains, its success on one domain suggests deeper compositional ability rather than local memorization.

- Core principle 4: Robustness to distractors and paraphrasing
  - A memorized pattern is brittle under paraphrase or adversarial distractors; a reasoning model should still arrive at the correct output when prompts are paraphrased or phrased with distractors.
  - Why this reduces false negatives: Paraphrasing tests the underlying reasoning rather than superficial cues.

- Core principle 5: Verifiable, bounded reasoning trace (final answer with a concise justification)
  - Requiring a concise justification (without exposing chain-of-thought internals) makes it easier to differentiate if the model computed from rules versus repeating a memorized phrase. For evaluation, collect the final answer and a short justification that maps to a minimal set of steps or rules.
  - Why this reduces both false positives and negatives: A model that truly reasons must be able to cite the relevant rules or steps succinctly; a memorizer may provide generic or tautological justifications.

2) Scoring metrics, thresholds, and an overall pass/fail rubric

- Metrics per item
  - Correctness (CR): 1 if the final answer is correct; 0 otherwise.
  - Demonstrated generalization (DG): 1 if the answer uses correct generalizable reasoning (or a correct application of a general rule) in a way that would hold under a paraphrase or a slight variant; 0 otherwise.
  - Consistency under paraphrase (CP): 1 if the model produces a correct final answer under a specified paraphrase variant; 0 otherwise.
  - Conciseness and justification quality (CQ): 0–1 scale based on whether the justification is succinct, correct, and points to the right rules/steps.
  - Robustness to distractors (RD): 1 if correct answer remains after adversarial or distractor-laden prompt; 0 otherwise.

- Aggregate scoring
  - Item-level score: S_item = CR * 0.6 + DG * 0.25 + CP * 0.05 + CQ * 0.05 (weights can be tuned; core is correctness and demonstrated generalization)
  - Overall score: S_total = average(S_item) across all N items.
  - Pass threshold: A conservative pass might use S_total >= 0.75 with each item having CR=1 or DG=1 in at least some paraphrase; or, more stringently, require at least 6/8 items with CR=1 and the rest with DG=1.
  - Statistical confidence: Report 95% CIs for S_total using bootstrap resampling over runs/prompts. Use a McNemar test for pairwise model comparisons on item-level correctness when comparing two models on the same prompts.

- Decision rubric
  - Pass: S_total >= 0.75 and at least 6 of 8 items demonstrate non-memorized-like behavior (CR=1 or robust DG in paraphrase).
  - Borderline: S_total in [0.65, 0.75], investigate items with low DG/CP.
  - Fail: S_total < 0.65 or more than two items show only memorization-like outputs (CR=0, DG=0, CP=0).

3) Eight concrete test prompts (domains include multi-step symbolic reasoning, novel analogies, counterfactual/hypothetical reasoning, commonsense physics, planning under constraints, program synthesis using an invented API, and adversarial distractors)

For each item, you’ll see:
- Prompt text
- Example correct output (canonical)
- Brief rationale why diagnostic for compositional/generalization
- Brief notes on how to interpret results (observable differences between a memorizing model vs a generalizing model)

Test item 1 — Multi-step symbolic reasoning
- Prompt:
  You are given the system:
    x + y = 7
    x − y = 3
  Compute x^2 + y^2.
- Example correct output:
  29
- Rationale / diagnostic:
  Requires solving a small linear system and computing a derived quantity. A memorization-based model would be unlikely to reliably reuse an internal solving pattern for new numeric facts; a reasoning model will consistently produce the correct result through the two-step deduction and calculation.
- Expected observable differences:
  A (memorizer): may output 29 but with weak justification or fail under paraphrase (e.g., “29 is the answer because it's a square of something”). B (reasoner): outputs 29 with concise justification like “From x=5, y=2; x^2+y^2=25+4=29.”

Test item 2 — Novel analogy
- Prompt:
  Provide a concise original analogy that expresses the relation: “the tool enables a task” in the form: X is to Y as Z is to W. Use X=keyboard, Y=typing, and provide a plausible Z and W. One acceptable output: “brush is to painting” as a parallel domain analogy.
- Example correct output:
  Keyboard is to typing as brush is to painting.
- Rationale / diagnostic:
  Tests ability to generate a structurally valid, domain-transcending analogy rather than regurgitate a common meme. A memorizer would likely reproduce a common cliche rather than a new, principled analogy.
- Expected observable differences:
  A (memorizer): repeats a very common analogy (e.g., “keyboard is to typing as pen is to writing”) without adapting to the domain shift. B (reasoner): produces a more novel but coherent analogy that preserves the relation and demonstrates cross-domain mapping.

Test item 3 — Counterfactual and hypothetical reasoning
- Prompt:
  Suppose Earth’s gravity suddenly doubled for all objects. What would change about (a) your weight, and (b) the acceleration of a freely falling object near the surface? Provide concise final statements.
- Example correct output:
  (a) Your weight would double (mass unchanged; weight = mass × g). (b) The acceleration of a freely falling object would double to 19.6 m/s^2.
- Rationale / diagnostic:
  Tests ability to reason about a hypothetical physical change and apply known relationships (weight scales with g; acceleration of gravity depends on g). A memorizer might rely on memorized physics phrases; a generalizer will derive the relationships.
- Expected observable differences:
  A (memorizer): might give a plausible but incorrect claim (e.g., “everything becomes heavier but gravity stays the same”). B (reasoner): correct, with explicit mention of mass, weight, and g scaling.

Test item 4 — Commonsense physics
- Prompt:
  A hammer is dropped on the Moon from the same height as on Earth (no atmosphere). Do the hammer and a feather hit the ground at different times? Explain briefly.
- Example correct output:
  They hit at the same time (ignoring air resistance, Moon has no atmosphere, so all objects accelerate at the same rate under Moon’s gravity).
- Rationale / diagnostic:
  Tests counterintuitive physics under atmospheric conditions. Requires the model to apply a general principle (free-fall in vacuum) rather than recall a memorized fact.
- Expected observable differences:
  A (memorizer): may claim obvious Earth-like behavior with air resistance differences. B (reasoner): states the vacuum principle and the Moon’s gravity.

Test item 5 — Planning under constraints
- Prompt:
  You must schedule three tasks within a 6-hour window:
  - A: duration 2h
  - B: duration 3h, depends on A (B can start only after A finishes)
  - C: duration 1h, independent
  Provide a feasible schedule with start times (in hours from window start 0). If impossible, state so.
- Example correct output:
  A: 0–2, B: 2–5, C: 5–6
- Rationale / diagnostic:
  Tests planning with dependencies and tight resource constraints. A memorizer might memorize a typical scheduling pattern; a generalizer will compute a legal schedule or prove infeasibility by reasoning about constraints.
- Expected observable differences:
  A (memorizer): outputs a schedule that may violate dependency or total window if forced into a template; B (reasoner): provides a consistent, constraint-satisfying plan.

Test item 6 — Program synthesis using an invented API
- Prompt (invented API description):
  Suppose the following API is available:
  - api.make_list(n, f): returns [f(0), f(1), ..., f(n-1)]
  - api.map(list, func): returns [func(x) for x in list]
  - api.filter(list, predicate): returns [x for x in list if predicate(x)]
  - api.sum(list): returns the sum of numbers in list
  - api.sort(list, key=None): returns a sorted copy
  Your task: write code using the above API to compute the sum of squares of even numbers in the range 0..9 (inclusive).
- Example correct output (code):
  def compute():
      nums = api.make_list(10, lambda i: i)        # [0,1,2,3,4,5,6,7,8,9]
      evens = api.filter(nums, lambda n: n % 2 == 0)  # [0,2,4,6,8]
      squares = api.map(evens, lambda n: n*n)      # [0,4,16,36,64]
      return api.sum(squares)                      # 120
- Rationale / diagnostic:
  Demonstrates algorithmic composition using a small API; requires understanding when to map/ filter/ sum in sequence, not memorized templates. A memorizing model might output a plausible code snippet but fail to use the API coherently or produce incorrect composition.
- Expected observable differences:
  A (memorizer): code that looks plausible but misuses API names or misordered operations, or returns an incorrect result. B (reasoner): correct API usage and final output 120.

Test item 7 — Adversarial distractors
- Prompt:
  Solve the simple algebraic equation, but the prompt contains distractor text to tempt pattern matching:
  "In a linear world, if 3x + 5 = 20, determine x. Note: ignore the background noise and focus on the arithmetic." Options: (A) 3, (B) 5, (C) 4, (D) 7. Explain why your answer is correct.
- Example correct output:
  x = 5. Correct option: (B).
- Rationale / diagnostic:
  Tests resilience to distractors and reliance on actual algebra rather than template-based extraction or pattern matching from training data. A memorizer might misinterpret the distractor or reproduce a common wrong option.
- Expected observable differences:
  A (memorizer): selects a distractor consistently due to pattern familiarity; B (reasoner): justifies algebraically and picks 5, with a short justification (e.g., “subtract 5, then divide by 3”).

Test item 8 — Formal rewriting / symbol manipulation (non-trivial generalization)
- Prompt:
  Apply the following two rewriting rules to the initial string S = "abcbc":
  Rule 1: replace "ab" with "c"
  Rule 2: replace "bc" with "d"
  Apply the rules left-to-right, repeatedly, until no more replacements are possible. What is the final string?
- Example correct output:
  "ccd"
- Rationale / diagnostic:
  Tests the ability to execute a small formal rewriting system and compose multiple steps; counters memorization since the specific string and rules are simple but the exact computed result is sensitive to the order and repetition. A memorization-based model may guess or improvise; a reasoning model will apply rules deterministically.
- Expected observable differences:
  A (memorizer): may produce an incorrect final string or claim “ccd” due to pattern familiarity without tracing the steps; B (reasoner): correctly computes the sequence of rule applications and reports "ccd" with a brief justification (e.g., Step 1: ab→c; Step 2: bc→d; Step 3: no more matches; final: ccd).

4) Likely failure modes for memorizing models and concrete countermeasures

- Pattern-matching overfit
  - Failure mode: The model outputs the right answers for items that resemble memorized templates but fails on novel paraphrases or minor variant inputs.
  - Countermeasures: Introduce paraphrased variants of prompts and randomize prompt wording, so that success cannot rely on a fixed template. Require correctness under at least one paraphrase per item.

- Template over-reliance (surface cues)
  - Failure mode: The model leans on generic, template-like justification that sounds plausible but does not reflect actual computation.
  - Countermeasures: Collect only the final answer plus a concise justification; explicitly evaluate whether the justification references correct rules/steps. Include an item that uses a different rule set than seen in training data to force genuine rule application.

- Memorized ingredient replacement
  - Failure mode: For program-synthesis, the model outputs code that appears syntactically valid but uses API names in non-existent ways or misuses the API, exploiting memorized patterns of code structure.
  - Countermeasures: Use a controlled API with explicit, minimal semantics (as in Test item 6) and require that the code run in a simple interpreter or be verifiable against the API’s semantics.

- Adversarial distractors that mimic training data
  - Failure mode: The model latches onto distractor sentences that resemble training data and anchors on them.
  - Countermeasures: Include adversarial variants and require robust evaluation with multiple distractor types (lexical, syntactic, and semantic distractors) and measure stability of outputs.

- Domain-specific blind spots
  - Failure mode: Some domain-specific prompts (especially hypothetical physics or planning) could reveal domain blind spots.
  - Countermeasures: Ensure variety across domains (as in the eight items) and include cross-domain prompts to test generalization rather than domain-specific memorization.

5) Evaluation protocol (fairness, reproducibility, black-box text I/O)

- Model access and setup
  - Use identical prompt sequences across models; keep the prompt content fixed while randomizing minor wording across paraphrase variants as part of the protocol (see paraphrase plan below).
  - Temperature and sampling: Run each prompt at multiple settings (e.g., T ∈ {0.0, 0.2, 0.5}) to test stability and reduce stochastic artifacts. Record the best-performing setting per item.
  - Runs per item: At least 4 independent runs per model per paraphrase (different seeds/temperatures) to estimate variance. Report mean and 95% CI for each metric (CR, DG, CP, CQ, RD).

- Randomization and prompt paraphrasing
  - For each item, generate at least two paraphrase variants that preserve the same underlying task (e.g., different surface wording, different numeric instantiation for the same relations).
  - Randomize the order of items within a test batch to prevent any model from discerning a fixed order.

- Scoring and statistics
  - Per-item scoring uses the weighted rubric described above (CR, DG, CP, CQ, RD).
  - Compute overall S_total and 95% bootstrap CIs.
  - When comparing two models, apply McNemar’s test on per-item correctness to determine if the difference in success rate is statistically significant.

- Reproducibility and reporting
  - Publish the exact eight prompts (with their canonical correct outputs) and the invented API specification used in Test item 6.
  - Report per-item observed accuracies, paraphrase robustness, and per-item justification quality.
  - Provide a reproducibility appendix: random seeds, paraphrase variants, temperatures, and run counts.

- Fairness considerations
  - Use the same evaluation hardware or ensure latency and token limits are not advantaging any model.
  - If models have different token-length budgets, clip or normalize prompts to a maximal reasonable length so that no model can rely on longer-context advantages.

6) Per-item observable differences to aid external researchers

For each test item, researchers can compare outputs from a memorizing model vs a genuinely reasoning model:

- Test item 1 (multi-step reasoning)
  - Memorizer A: Final numeric answer 29 but justification vague or generic; may fail under paraphrase.
  - Reasoner B: Final answer 29 with a minimal, correct two-step justification (solve x and y, compute x^2+y^2) that generalizes to changed numbers.

- Test item 2 (novel analogy)
  - Memorizer A: Reproduces a common analogy without adaptation (e.g., keyboard is to typing as pen is to writing) when the prompt emphasizes novelty.
  - Reasoner B: Produces a coherent, original analogy preserving the relation, and explains the relation succinctly.

- Test item 3 (counterfactual reasoning)
  - Memorizer A: Returns a plausible-sounding statement that might be wrong (e.g., weight changes but acceleration unchanged).
  - Reasoner B: Correctly explains both weight doubling and acceleration under doubled gravity; paraphrase variant retains correctness.

- Test item 4 (commonsense physics)
  - Memorizer A: Claims the feather falls first due to some surface interaction; depends on Earth-bias.
  - Reasoner B: Correct vacuum intuition: in vacuum, the hammer and feather hit ground simultaneously on the Moon; states vacuum principle clearly.

- Test item 5 (planning under constraints)
  - Memorizer A: Schedules that violate dependency or length constraint on some paraphrase.
  - Reasoner B: Produces a dependency-respecting schedule that fits all constraints.

- Test item 6 (program synthesis with invented API)
  - Memorizer A: Produces code that looks plausible but uses API in inconsistent ways or misses a step (e.g., uses map without filtering even numbers).
  - Reasoner B: Produces a correct, fully coherent sequence using make_list, filter, map, and sum; code runs to produce 120.

- Test item 7 (adversarial distractors)
  - Memorizer A: Picks a distractor (e.g., option C) due to memorized patterns, regardless of actual algebra.
  - Reasoner B: Solves the equation 3x+5=20 and selects option B (x=5) with a short justification.

- Test item 8 (formal rewriting)
  - Memorizer A: Might misapply the rules or produce an alternative final string (e.g., “bd” or “ccd” with incorrect intermediate steps) depending on memorized patterns.
  - Reasoner B: Correctly applies Rule 1 and Rule 2 in left-to-right order and returns “ccd” with a brief justification of the rule applications.

7) Summary and practical recommendations

- This evaluation suite is designed to stress-test compositional reasoning by combining classic math-style problems, novel analogies, hypothetical physics, planning, program synthesis, and formal rewriting. It emphasizes:
  - Novelty and combinatorial reasoning
  - Paraphrase and distractor robustness
  - Cross-domain generalization
  - Minimal, targeted justification rather than chain-of-thought leakage

- If a model consistently scores high on tests that require memorization (identical prompts or heavily templated patterns) but fails on paraphrased or transformed variants, it indicates memorization bias rather than genuine reasoning.

- Conversely, a model that maintains correctness across paraphrases, performs consistent multi-step transformations (e.g., code composition, formal rewrites, planning under constraints) and gives concise, rule-based justifications is indicative of compositional generalization.

- Practical deployment notes:
  - Start with 4–6 paraphrase variants per item to gauge robustness; extend to 2–3 more for a final evaluation batch.
  - Use multiple temperatures and report the best-performing setting per item along with a stability analysis (variance across runs).
  - Consider releasing the eight prompts with canonical answers and a scoring rubric to enable independent replication.

If you’d like, I can provide:
- A ready-to-run evaluation script (pseudo-Python) that encodes these eight prompts, paraproparaphrase variants, and the scoring rubric.
- A per-item rubric rubric file (CSV/JSON) specifying CR, DG, CP, CQ, RD criteria and weightings.
- A small synthetic dataset of paraphrase variants for each item to kick off reproducible experiments.

In [10]:
# Anthropic has a slightly different API, and Max Tokens is required

model_name = "claude-sonnet-4-5"

claude = Anthropic()
response = claude.messages.create(model=model_name, messages=messages, max_tokens=1000)
answer = response.content[0].text

display(Markdown(answer))
competitors.append(model_name)
answers.append(answer)

TypeError: "Could not resolve authentication method. Expected either api_key or auth_token to be set. Or for one of the `X-Api-Key` or `Authorization` headers to be explicitly omitted"

In [ ]:
gemini = OpenAI(api_key=google_api_key, base_url="https://generativelanguage.googleapis.com/v1beta/openai/")
model_name = "gemini-2.5-flash"

response = gemini.chat.completions.create(model=model_name, messages=messages)
answer = response.choices[0].message.content

display(Markdown(answer))
competitors.append(model_name)
answers.append(answer)

In [ ]:
deepseek = OpenAI(api_key=deepseek_api_key, base_url="https://api.deepseek.com/v1")
model_name = "deepseek-chat"

response = deepseek.chat.completions.create(model=model_name, messages=messages)
answer = response.choices[0].message.content

display(Markdown(answer))
competitors.append(model_name)
answers.append(answer)

In [ ]:
# Updated with the latest Open Source model from OpenAI

groq = OpenAI(api_key=groq_api_key, base_url="https://api.groq.com/openai/v1")
model_name = "openai/gpt-oss-120b"

response = groq.chat.completions.create(model=model_name, messages=messages)
answer = response.choices[0].message.content

display(Markdown(answer))
competitors.append(model_name)
answers.append(answer)


## For the next cell, we will use Ollama

Ollama runs a local web service that gives an OpenAI compatible endpoint,  
and runs models locally using high performance C++ code.

If you don't have Ollama, install it here by visiting https://ollama.com then pressing Download and following the instructions.

After it's installed, you should be able to visit here: http://localhost:11434 and see the message "Ollama is running"

You might need to restart Cursor (and maybe reboot). Then open a Terminal (control+\`) and run `ollama serve`

Useful Ollama commands (run these in the terminal, or with an exclamation mark in this notebook):

`ollama pull <model_name>` downloads a model locally  
`ollama ls` lists all the models you've downloaded  
`ollama rm <model_name>` deletes the specified model from your downloads

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/stop.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#ff7800;">Super important - ignore me at your peril!</h2>
            <span style="color:#ff7800;">The model called <b>llama3.3</b> is FAR too large for home computers - it's not intended for personal computing and will consume all your resources! Stick with the nicely sized <b>llama3.2</b> or <b>llama3.2:1b</b> and if you want larger, try llama3.1 or smaller variants of Qwen, Gemma, Phi or DeepSeek. See the <A href="https://ollama.com/models">the Ollama models page</a> for a full list of models and sizes.
            </span>
        </td>
    </tr>
</table>

In [ ]:
!ollama pull llama3.2

In [ ]:
ollama = OpenAI(base_url='http://localhost:11434/v1', api_key='ollama')
model_name = "llama3.2"

response = ollama.chat.completions.create(model=model_name, messages=messages)
answer = response.choices[0].message.content

display(Markdown(answer))
competitors.append(model_name)
answers.append(answer)

In [ ]:
# So where are we?

print(competitors)
print(answers)


In [ ]:
# It's nice to know how to use "zip"
for competitor, answer in zip(competitors, answers):
    print(f"Competitor: {competitor}\n\n{answer}")


In [ ]:
# Let's bring this together - note the use of "enumerate"

together = ""
for index, answer in enumerate(answers):
    together += f"# Response from competitor {index+1}\n\n"
    together += answer + "\n\n"

In [ ]:
print(together)

In [ ]:
judge = f"""You are judging a competition between {len(competitors)} competitors.
Each model has been given this question:

{question}

Your job is to evaluate each response for clarity and strength of argument, and rank them in order of best to worst.
Respond with JSON, and only JSON, with the following format:
{{"results": ["best competitor number", "second best competitor number", "third best competitor number", ...]}}

Here are the responses from each competitor:

{together}

Now respond with the JSON with the ranked order of the competitors, nothing else. Do not include markdown formatting or code blocks."""


In [ ]:
print(judge)

In [ ]:
judge_messages = [{"role": "user", "content": judge}]

In [ ]:
# Judgement time!

openai = OpenAI()
response = openai.chat.completions.create(
    model="gpt-5-mini",
    messages=judge_messages,
)
results = response.choices[0].message.content
print(results)


In [ ]:
# OK let's turn this into results!

results_dict = json.loads(results)
ranks = results_dict["results"]
for index, result in enumerate(ranks):
    competitor = competitors[int(result)-1]
    print(f"Rank {index+1}: {competitor}")

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/exercise.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#ff7800;">Exercise</h2>
            <span style="color:#ff7800;">Which pattern(s) did this use? Try updating this to add another Agentic design pattern.
            </span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/business.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#00bfff;">Commercial implications</h2>
            <span style="color:#00bfff;">These kinds of patterns - to send a task to multiple models, and evaluate results,
            are common where you need to improve the quality of your LLM response. This approach can be universally applied
            to business projects where accuracy is critical.
            </span>
        </td>
    </tr>
</table>